In [2]:
#1.	Load the given CSV file containing text and label columns into a Pandas DataFrame.
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP/cleaned_updated_data.csv')

Mounted at /content/drive


In [3]:
#A1. Data Loading
#Load the dataset into a Pandas DataFrame. Extract the text column and treat each entry as a separate document.

documents = df['review_text']
print(f"Number of documents: {len(documents)}")
print(f"First 5 documents:\n{documents.head()}")

Number of documents: 100
First 5 documents:
0    Taste of the food is nice but the dining and h...
1    Seemed to be very dark in the catalogue but wh...
2    Material is not nice but I like print and patt...
3    The quality is not anything that I'd put up in...
4    Durability is a concern but apart from that al...
Name: review_text, dtype: object


In [4]:
#A2. Tokenization
#Tokenize all documents and store the tokens corresponding to each document.
#You may use the NLTK library for tokenization.

import nltk
from nltk.tokenize import word_tokenize

# Download the 'punkt' tokenizer models required for word_tokenize
nltk.download('punkt_tab')

# Tokenize each document
tokenized_documents = [word_tokenize(doc) for doc in documents.astype(str)]

print(f"Number of tokenized documents: {len(tokenized_documents)}")
print(f"First 5 tokenized documents:\n{tokenized_documents[:5]}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Number of tokenized documents: 100
First 5 tokenized documents:
[['Taste', 'of', 'the', 'food', 'is', 'nice', 'but', 'the', 'dining', 'and', 'hygiene', 'of', 'the', 'restaurant', 'was', 'not', 'nice'], ['Seemed', 'to', 'be', 'very', 'dark', 'in', 'the', 'catalogue', 'but', 'when', 'arrived', 'the', 'color', 'fades', 'after', 'washing'], ['Material', 'is', 'not', 'nice', 'but', 'I', 'like', 'print', 'and', 'pattern', ',', 'for', 'this', 'cost', 'it', 'worth', 'only'], ['The', 'quality', 'is', 'not', 'anything', 'that', 'I', "'d", 'put', 'up', 'in', 'my', 'home', 'to', 'try'], ['Durability', 'is', 'a', 'concern', 'but', 'apart', 'from', 'that', 'all', 'other', 'factors', 'are', 'good']]


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
#A3. Token Population
#Merge the tokens obtained from all documents and create a master list of distinct tokens present across the entire dataset.
#Refer to this list as the token population.

# Flatten the list of tokenized documents into a single list of all tokens
all_tokens = [token for doc_tokens in tokenized_documents for token in doc_tokens]

# Create a master list of distinct tokens (token population) using a set for uniqueness
token_population = sorted(list(set(all_tokens)))

print(f"Total number of tokens: {len(all_tokens)}")
print(f"Number of distinct tokens (token population): {len(token_population)}")
print(f"First 20 tokens in population:\n{token_population[:20]}")

Total number of tokens: 669
Number of distinct tokens (token population): 315
First 20 tokens in population:
["'d", ',', '.', 'Actign', 'Amazing', 'BUY', 'Customer', 'DO', 'Delivery', 'Doesn', 'Durability', 'Ever', 'Everything', 'Excellent', 'Good', 'Great', 'Highly', 'I', 'It', 'Material']


In [6]:
#A4. Stop-words
# •	Study the concept of stop-words and identify why they are removed in text analysis.
"""Stop-words are common words (like 'the', 'is', 'and') that often carry little significant meaning in the context of text analysis
and can be removed to reduce noise and improve processing efficiency."""
# •	Load the English stop-words list using the NLTK library.
# •	Examine the stop-words and briefly state what kind of words they represent.
""" Stop-words typically represent functional words or grammatical connectors. They are often articles, prepositions, conjunctions, pronouns, and some common verbs or adverbs.
Their primary role is to construct sentences grammatically rather than to convey specific semantic content."""

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

english_stop_words = set(stopwords.words('english'))

print(f"Number of English stop-words: {len(english_stop_words)}")
print("\nFirst 20 English stop-words (alphabetical order):\n")
for i, word in enumerate(sorted(list(english_stop_words))) :
    if i >= 20:
        break
    print(word, end = ", ")

[nltk_data] Downloading package stopwords to /root/nltk_data...


Number of English stop-words: 198

First 20 English stop-words (alphabetical order):

a, about, above, after, again, against, ain, all, am, an, and, any, are, aren, aren't, as, at, be, because, been, 

[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
#A5. Bag-of-Words Construction
# •	Remove stop-words from the token population obtained in A3.
# •	Construct a Bag-of-Words (BoW) using the remaining tokens.
# •	The Bag-of-Words should consist only of unique, meaningful tokens.

filtered_tokens = [token for token in token_population if token.lower() not in english_stop_words]

bag_of_words_tokens = sorted(list(set(filtered_tokens)))

print(f"Number of tokens in Bag-of-Words (after stop-word removal): {len(bag_of_words_tokens)}")
print(f"First 20 tokens in Bag-of-Words:\n{bag_of_words_tokens[:20]}")

Number of tokens in Bag-of-Words (after stop-word removal): 253
First 20 tokens in Bag-of-Words:
["'d", ',', '.', 'Actign', 'Amazing', 'BUY', 'Customer', 'Delivery', 'Durability', 'Ever', 'Everything', 'Excellent', 'Good', 'Great', 'Highly', 'Material', 'Nice', 'Nothing', 'Overall', 'Premium']


In [8]:
#A6. Document Vectorization
#For each document, create a Bag-of-Words feature vector based on the vocabulary constructed in A5.
# •	Each vector dimension corresponds to a word in the Bag-of-Words.
# •	The value in each dimension represents the count of occurrences of that word in the document.
# After this step, all documents should be represented in vector form using the same Bag-of-Words vocabulary.

from collections import Counter

# Create a dictionary to map each word in bag_of_words_tokens to its index
vocabulary_index_map = {word: i for i, word in enumerate(bag_of_words_tokens)}

# Initialize a list to store document vectors
document_vectors = []

# Process each tokenized document
for doc_tokens in tokenized_documents:
    # Count word occurrences in the current document
    word_counts = Counter(doc_tokens)

    # Create a vector for the current document
    # The vector's size is the size of our BoW vocabulary
    document_vector = [0] * len(bag_of_words_tokens)

    # Populate the vector with word counts
    for word, count in word_counts.items():
        # Only consider words that are in our Bag-of-Words vocabulary
        if word in vocabulary_index_map:
            index = vocabulary_index_map[word]
            document_vector[index] = count
    document_vectors.append(document_vector)

print(f"Number of document vectors: {len(document_vectors)}")
print(f"Shape of the first document vector: {len(document_vectors[0])}")
print(f"First document vector (first 20 elements): {document_vectors[0][:20]}")

Number of document vectors: 100
Shape of the first document vector: 253
First document vector (first 20 elements): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [9]:
#A7. Dataset Preparation for Classification
# •	Combine the document vectors with their corresponding sentiment labels.
# •	Split the dataset into training and testing sets.

from sklearn.model_selection import train_test_split
import numpy as np

# Extract sentiment labels from the original DataFrame
# Ensure 'sentiment' column exists and handle potential NaN values if necessary
labels = df['sentiment']

# Convert document_vectors to a NumPy array for easier handling with sklearn
X_original = np.array(document_vectors)

# Filter out rows where sentiment labels are NaN
# Create a boolean mask for non-NaN labels
valid_mask = ~labels.isna()

# Apply the mask to both the document vectors (X) and labels (y) to ensure they remain aligned
X = X_original[valid_mask]
y = labels[valid_mask].to_numpy() # Convert filtered Series to numpy array

# Split the dataset into training and testing sets
# We'll use a common split ratio, e.g., 80% for training and 20% for testing
# stratify=y ensures that the proportion of sentiment labels is maintained in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# Display the first few labels of the training set to verify
print(f"\nFirst 5 y_train labels: {y_train[:5]}")
print(f"\nFirst 5 y_test labels: {y_test[:5]}")

Shape of X_train: (76, 253)
Shape of X_test: (19, 253)
Shape of y_train: (76,)
Shape of y_test: (19,)

First 5 y_train labels: ['positive' 'positive' 'positive' 'positive' 'negative']

First 5 y_test labels: ['positive' 'negative' 'negative' 'negative' 'neutral']


In [10]:
#A8. Sentiment Classification
# •	Train a machine learning classifier of your choice (e.g., Naive Bayes or Logistic Regression) using the Bag-of-Words vectors.
# •	Use the trained model to predict the sentiment labels for the test data

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize a Random Forest Classifier
# Setting random_state for reproducibility
classifier = RandomForestClassifier(random_state=42)

# Train the classifier on the training data
classifier.fit(X_train, y_train)

# Predict the sentiment labels for the test data
y_pred = classifier.predict(X_test)


In [11]:
#A9. Model Evaluation
# •	Evaluate the performance of the classifier using appropriate metrics such as accuracy, precision, recall, or confusion matrix.
# •	Briefly comment on the classification results.

print("Sentiment Classification Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# The classification results (Accuracy and Classification Report) were already printed in cell A8.
# Let's interpret them here based on the output from the RandomForestClassifier.

# --- Interpretation of Classification Results ---
# The overall Accuracy for the RandomForestClassifier is 0.3158 (or 31.58%). This is quite low,
# indicating that the model struggles to correctly classify a majority of the test samples.

# Looking at the Classification Report:
# - Precision: This metric tells us, for each class, out of all instances predicted as that class, how many were actually correct.
#   - For 'negative' class: 0.00. This means that out of all samples the model *predicted* as negative, none were actually negative.
#     This usually happens when the model did not predict any samples as 'negative' (as shown by support=6 and precision=0.00 with UndefinedMetricWarning).
#   - For 'neutral' class: 0.33. Only 33% of the samples predicted as neutral were actually neutral.
#   - For 'positive' class: 0.31. Only 31% of the samples predicted as positive were actually positive.

# - Recall: This metric tells us, for each class, out of all actual instances of that class, how many were correctly identified.
#   - For 'negative' class: 0.00. The model failed to identify any of the actual negative samples.
#   - For 'neutral' class: 0.17. It correctly identified only 17% of the actual neutral samples.
#   - For 'positive' class: 0.71. It performed best on the positive class, correctly identifying 71% of the actual positive samples.

# - F1-score: This is the harmonic mean of precision and recall, providing a single metric that balances both.
#   - The F1-scores are very low for 'negative' (0.00) and 'neutral' (0.22), and only moderate for 'positive' (0.43).

# - Support: This indicates the number of actual occurrences of each class in the test set.
#   - Negative: 6 samples
#   - Neutral: 6 samples
#   - Positive: 7 samples

# - Macro Avg: The average of the unweighted per-class metrics.
# - Weighted Avg: The average of the per-class metrics, weighted by their support (number of true instances for each class).

# Overall Comment:
# The `RandomForestClassifier` shows poor performance on this sentiment classification task with the current Bag-of-Words features.
# The model struggles significantly with the 'negative' and 'neutral' classes, often failing to predict them at all (indicated by 0.00 precision and recall for 'negative').
# It performs somewhat better on the 'positive' class in terms of recall, but its precision is still low.
# This suggests that the current feature representation (simple Bag-of-Words without further refinement) or the model's parameters
# are not sufficient for accurately distinguishing between these sentiment classes in this dataset. Further preprocessing steps
# (e.g., stemming/lemmatization, TF-IDF, N-grams) or trying other classifiers/hyperparameter tuning might be necessary to improve performance.


Sentiment Classification Results:
Accuracy: 0.3158

Classification Report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         6
     neutral       0.33      0.17      0.22         6
    positive       0.31      0.71      0.43         7

    accuracy                           0.32        19
   macro avg       0.22      0.29      0.22        19
weighted avg       0.22      0.32      0.23        19



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
